Ce notebook génère **3 fichiers CSV** utilisés par `ComparaisonCS_final.ipynb` :
| Fichier | Contenu | Nb features |
|---|---|---|
| `dfbase.csv` | Variables d'origine + encodage quali | ~p |
| `dfpoly.csv` | dfbase + termes X² et X³ | ~2p |
| `dfinter.csv` | dfbase + interactions d'ordre 2 | ~p(p-1)/2 |
| `dffull.csv` | dfbase + termes X² et X³ + interactions d'ordre 2 | ~p(p+5)/2 |

> ⚙️ **Chemin de sauvegarde** : modifiez `SORTIE` dans la cellule suivante si nécessaire.

In [74]:
import pandas as pd
import numpy as np
from patsy import dmatrix

# Chemin de sortie des CSV (relatif à ce notebook)
# Par défaut : FICHIERS FINAUX/ (là où ComparaisonCS_final.ipynb les attend)
SORTIE = "../Session3/"

## 1. Chargement et exploration

In [75]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/SAh.csv",
                  header=0, sep=",")
### import fichier csv
# don = pd.read_csv("spambase/spambase.data", header=None, sep=",")

### import fichier texte :
# don = pd.read_table("ozone.txt", header=0, sep=";")

don.head()

,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,chd
0,160,12.00,5.73,23.11,Present,49,25.30,97.20,52,1
1,144,0.01,4.41,28.61,Absent,55,28.87,2.06,63,1
2,118,0.08,3.48,32.28,Present,52,29.14,3.81,46,0
3,170,7.50,6.41,38.03,Present,51,31.99,24.26,58,1
4,134,13.60,3.50,27.78,Present,60,25.99,57.34,49,1


In [76]:
# Si fichier de données sans noms de colonnes
# 2. Renommer la première colonne en "Y"
# column_names = [f'X{i}' for i in range(len(don.columns) - 1)] + ['Y']
# don.columns = column_names
# don.head()

In [63]:
# Si on souhaite retirer des variables inutiles du df (ex : date)
don = don.drop(columns=["Date"])
don.head(3)

,O3,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite,vent
0,63.6,13.4,15.0,7,0,0,3,0,9.35,95.6,NUAGE,EST
1,89.6,15.0,15.7,4,3,0,0,0,5.40,100.2,SOLEIL,NORD
2,79.0,7.9,10.1,8,0,0,7,0,19.30,105.6,NUAGE,EST


In [77]:
# Modifier nom variable à prédire
don.rename(columns={"chd": "Y"}, inplace=True)
print(f"Dimensions      : {don.shape[0]} observations x {don.shape[1]} colonnes")
print(f"Proportion Y=1  : {don.Y.mean():.1%} ({int(don.Y.sum())} malades / {len(don)} individus)")
print(f"Types de variables :")
print(don.dtypes)
don.describe(include='all')

Dimensions      : 462 observations x 10 colonnes
Proportion Y=1  : 34.6% (160 malades / 462 individus)
Types de variables :
sbp            int64
tobacco      float64
ldl          float64
adiposity    float64
famhist       object
typea          int64
obesity      float64
alcohol      float64
age            int64
Y              int64
dtype: object


,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,Y
count,462.000000,462.000000,462.000000,462.000000,462,462.000000,462.000000,462.000000,462.000000,462.000000
unique,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,Absent,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,270,NaN,NaN,NaN,NaN,NaN
mean,138.326840,3.635649,4.740325,25.406732,NaN,53.103896,26.044113,17.044394,42.816017,0.346320
std,20.496317,4.593024,2.070909,7.780699,NaN,9.817534,4.213680,24.481059,14.608956,0.476313
min,101.000000,0.000000,0.980000,6.740000,NaN,13.000000,14.700000,0.000000,15.000000,0.000000
25%,124.000000,0.052500,3.282500,19.775000,NaN,47.000000,22.985000,0.510000,31.000000,0.000000
50%,134.000000,2.000000,4.340000,26.115000,NaN,53.000000,25.805000,7.510000,45.000000,0.000000
75%,148.000000,5.500000,5.790000,31.227500,NaN,60.000000,28.497500,23.892500,55.000000,1.000000


## 2. Séparation X/Y et encodage des variables qualitatives

Je renomme la variable d'intérêt Y, je mets en général les X d'un côté et je regarde les types de variables afin de recoder les variables qualitatives

In [78]:
X = don.drop(columns=["Y"])
Y = don[["Y"]]

Xquanti = X.select_dtypes(exclude=['object'])
Xquali  = X.select_dtypes(include=['object'])
print("Variables quantitatives :", list(Xquanti.columns))
print("Variables qualitatives  :", list(Xquali.columns))

# Encodage one-hot (indicatrices) : drop_first=True évite la colinéarité (supprime la modalité de référence)
# Pour SAh : famhist (Present/Absent) -> famhist_Present = 1 ou 0
# Encodage one-hot (uniquement si Xquali n'est pas vide)
if not Xquali.empty:
    XqualiD = pd.get_dummies(Xquali, drop_first=True, dtype=float)
    print("Colonne(s) après encodage :", list(XqualiD.columns))
else:
    XqualiD = pd.DataFrame()  # DataFrame vide si aucune variable qualitative
    print("Aucune variable qualitative à encoder.")

# Base commune : quantitatives + qualitatives encodées
Xbase = pd.concat([Xquanti, XqualiD], axis=1)
print(f"\nNombre de variables dans Xbase : {Xbase.shape[1]}")

Xbase.head()

Variables quantitatives : ['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']
Variables qualitatives  : ['famhist']
Colonne(s) après encodage : ['famhist_Present']

Nombre de variables dans Xbase : 9


,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0
3,170,7.50,6.41,38.03,51,31.99,24.26,58,1.0
4,134,13.60,3.50,27.78,60,25.99,57.34,49,1.0


## 3. `dfbase.csv` — Variables d'origine encodées

In [79]:
dfbase = pd.concat([Xbase, Y], axis=1)
print(f"dfbase : {dfbase.shape[0]} obs x {dfbase.shape[1]-1} features + Y")
dfbase.head(3)

dfbase : 462 obs x 9 features + Y


,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present,Y
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0,1
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0,1
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0,0


In [80]:
dfbase.to_csv(SORTIE + "dfbase.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfbase.csv")
dfbase.head()

Sauvegarde OK : ../Session3/dfbase.csv


,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present,Y
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0,1
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0,1
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0,0
3,170,7.50,6.41,38.03,51,31.99,24.26,58,1.0,1
4,134,13.60,3.50,27.78,60,25.99,57.34,49,1.0,1


## 4. `dfpoly.csv` — Variables + termes polynomiaux (X², X³)
On ajoute les carrés et cubes des variables **quantitatives** uniquement.

In [81]:
X2 = Xquanti ** 2; X2 = X2.add_suffix("_sq")
X3 = Xquanti ** 3; X3 = X3.add_suffix("_cu")
Xpoly = pd.concat([Xbase, X2, X3], axis=1)
dfpoly = pd.concat([Xpoly, Y], axis=1)
print(f"dfpoly : {dfpoly.shape[0]} obs x {dfpoly.shape[1]-1} features + Y")
dfpoly.head(3)

dfpoly : 462 obs x 25 features + Y


,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present,sbp_sq,...,age_sq,sbp_cu,tobacco_cu,ldl_cu,adiposity_cu,typea_cu,obesity_cu,alcohol_cu,age_cu,Y
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0,25600,...,2704,4096000,1728.000000,188.132517,12342.406231,117649,16194.277000,918330.048000,140608,1
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0,20736,...,3969,2985984,0.000001,85.766121,23418.203381,166375,24062.478103,8.741816,250047,1
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0,13924,...,2116,1643032,0.000512,42.144192,33635.708352,140608,24743.927944,55.306341,97336,0


In [82]:
dfpoly.to_csv(SORTIE + "dfpoly.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfpoly.csv")

Sauvegarde OK : ../Session3/dfpoly.csv


## 5. `dfinter.csv` — Variables + interactions d'ordre 2
On ajoute tous les produits croisés entre variables via la formule patsy `(X1+X2+...)**2`.  
Cela modélise l'effet combiné de deux variables (ex. : âge × tabac).

In [83]:
nomsvar = list(don.columns.difference(["Y"]))
formuleI = "~1+(" + "+".join(nomsvar) + ")**2"  # effetsprincipaux et interactions d'ordre 2
Xinter = dmatrix(formuleI, don, return_type="dataframe").iloc[:, 1:]  # supprime l'intercept
dfinter = pd.concat([Xinter, Y], axis=1)
print(f"dfinter : {dfinter.shape[0]} obs x {dfinter.shape[1]-1} features + Y")
dfinter.head(3)

dfinter : 462 obs x 45 features + Y


,famhist[T.Present],adiposity,adiposity:famhist[T.Present],age,age:famhist[T.Present],alcohol,alcohol:famhist[T.Present],ldl,famhist[T.Present]:ldl,obesity,...,ldl:sbp,ldl:tobacco,ldl:typea,obesity:sbp,obesity:tobacco,obesity:typea,sbp:tobacco,sbp:typea,tobacco:typea,Y
0,1.0,23.11,23.11,52.0,52.0,97.20,97.20,5.73,5.73,25.30,...,916.80,68.7600,280.77,4048.00,303.6000,1239.70,1920.00,7840.0,588.00,1
1,0.0,28.61,0.00,63.0,0.0,2.06,0.00,4.41,0.00,28.87,...,635.04,0.0441,242.55,4157.28,0.2887,1587.85,1.44,7920.0,0.55,1
2,1.0,32.28,32.28,46.0,46.0,3.81,3.81,3.48,3.48,29.14,...,410.64,0.2784,180.96,3438.52,2.3312,1515.28,9.44,6136.0,4.16,0


In [84]:
dfinter.to_csv(SORTIE + "dfinter.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfinter.csv")

Sauvegarde OK : ../Session3/dfinter.csv


## 6. `dffull.csv` — Variables + termes carrés et cubiques + interactions d'ordre 2

In [85]:
# On récupère uniquement les colonnes nouvelles de chaque bloc (sans Y ni doublons)
cols_base  = dfbase.columns.difference(["Y"])
cols_poly  = dfpoly.columns.difference(["Y"]).difference(cols_base)   # nouveautés poly
cols_inter = dfinter.columns.difference(["Y"]).difference(cols_base)  # nouveautés inter

dffull = pd.concat([
    dfbase[cols_base],    # effets principaux
    dfpoly[cols_poly],    # X² et X³ uniquement (pas de doublons)
    dfinter[cols_inter],  # interactions uniquement (pas de doublons)
    Y                     # variable cible
], axis=1)

print(f"dffull : {dffull.shape[0]} obs x {dffull.shape[1]-1} features + Y")
dffull.to_csv("dffull.csv", index=False)

dffull : 462 obs x 62 features + Y


## Récapitulatif

In [86]:
recap = pd.DataFrame({
    "Fichier"     : ["dfbase.csv", "dfpoly.csv", "dfinter.csv", "dffull.csv"],
    "Nb features" : [dfbase.shape[1]-1, dfpoly.shape[1]-1,
                     dfinter.shape[1]-1, dffull.shape[1]-1],
    "Contenu"     : [
        "Variables d'origine encodées",
        "dfbase + termes quadratiques et cubiques",
        "dfbase + interactions d'ordre 2",
        "dfbase + polynômes + interactions (complet)"
    ]
})
print(recap.to_string(index=False))

    Fichier  Nb features                                     Contenu
 dfbase.csv            9                Variables d'origine encodées
 dfpoly.csv           25    dfbase + termes quadratiques et cubiques
dfinter.csv           45             dfbase + interactions d'ordre 2
 dffull.csv           62 dfbase + polynômes + interactions (complet)
